# Reference DreamerV3 on Pendulum-v1 (matched hyperparameters)

A/B test against our implementation. Runs the standard **NM512/dreamerv3-torch**
on **Pendulum-v1 (state-based obs)** with hyperparameters matched to ours, logging
`eval_return` and saving parity videos (real-env + imagination).

**Question:** does the standard library also diverge under our hyperparameters, or
does it learn a stable swing-up? If it converges stably, the divergence is a bug in
*our* code, not the algorithm/task.

> The reference is pinned to `gym==0.22.0` + `numpy<1.24`, which have **no wheels for
> Colab's Python 3.12**. So instead of touching Colab's system Python, we build an
> isolated **Python 3.11 venv with `uv`** and run everything through it. No runtime
> restart needed.

Run cells top to bottom. Set the GPU runtime: *Runtime → Change runtime type → GPU*.

## 1. Clone your repo

In [ ]:
# The benchmark lives on the 'benchmark' branch. For a private repo, use a token:
#   !git clone -b benchmark https://<TOKEN>@github.com/aritraban21/Dreamerv3.git
%cd /content
!rm -rf /content/Dreamerv3
!git clone -b benchmark https://github.com/aritraban21/Dreamerv3.git /content/Dreamerv3
%cd /content/Dreamerv3/benchmarks/dreamerv3-torch
!ls

## 2. Build an isolated Python 3.11 venv and install deps

`uv` downloads a managed CPython 3.11, then installs the lean Pendulum deps + the
reference's pinned torch (CUDA build, so the venv still uses the Colab GPU). Takes a
couple of minutes (torch is ~2.5 GB). **No restart needed** — we never touch system Python.

In [ ]:
!pip -q install uv
!uv venv --python 3.11 /content/venv311
# lean deps (numpy 1.23.5 + gym 0.22.0 now have wheels because this venv is py3.11)
!uv pip install --python /content/venv311 -r requirements-pendulum.txt
# GPU torch (matches the reference pin). cu121 wheels are compatible with Colab's driver.
!uv pip install --python /content/venv311 torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121

## 3. Sanity check the venv (Python 3.11, numpy 1.23, CUDA visible)

In [ ]:
!/content/venv311/bin/python -c "import sys,torch,numpy; print('py', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', numpy.__version__)"

## 4. Quick smoke (~1–2 min) — confirm it runs on this Colab

Tiny model, few steps. Expect `eval_return` printed and two mp4s written.

In [ ]:
!cd /content/Dreamerv3/benchmarks/dreamerv3-torch && SDL_VIDEODRIVER=dummy MPLBACKEND=Agg \
  /content/venv311/bin/python -u dreamer.py --configs pendulum --logdir ./logdir/smoke \
  --device cuda:0 --compile False --steps 130 --prefill 60 --pretrain 5 \
  --eval_every 120 --eval_episode_num 1 --batch_size 4 --batch_length 16 \
  --dyn_deter 64 --dyn_hidden 64 --units 64 --dyn_stoch 8 --dyn_discrete 8 \
  --imag_horizon 5 --render_video True --imag_video_horizon 8
!echo '--- videos ---'; ls -la /content/Dreamerv3/benchmarks/dreamerv3-torch/videos/pendulum 2>/dev/null

## 5. Full run — matched hyperparameters, 200k steps

The real comparison. Same env + hyperparameters as our runs (`dyn_deter=4096`,
`dyn_stoch=32×32`, `units=1024`, `imag_horizon=16`, `batch=16×64`, `train_ratio=512`,
`discount=0.997`, actor entropy `3e-4`, …). `--render_video True` saves a real-env +
imagination mp4 at every eval; stdout is teed to `run.log`. Takes **hours** on a T4 —
keep the tab alive. `--compile False` skips a long torch.compile warmup (no effect on results).

In [ ]:
!cd /content/Dreamerv3/benchmarks/dreamerv3-torch && SDL_VIDEODRIVER=dummy MPLBACKEND=Agg \
  /content/venv311/bin/python -u dreamer.py --configs pendulum --logdir ./logdir/pendulum_ref \
  --device cuda:0 --compile False --seed 0 --render_video True 2>&1 | tee run.log

## 6. Plot the eval-return curve

Runs in the Colab kernel (reads files written by the venv run). Reads `eval_return`
from `metrics.jsonl`. To overlay OUR run, upload one of our `Pendulum-v1_*.log` files
and set `OUR_LOG`.

In [ ]:
import json, re, pathlib
import matplotlib.pyplot as plt
BASE = '/content/Dreamerv3/benchmarks/dreamerv3-torch'

def ref_curve(metrics_path):
    steps, rets = [], []
    for line in pathlib.Path(metrics_path).read_text().splitlines():
        try: d = json.loads(line)
        except Exception: continue
        if 'eval_return' in d:
            steps.append(d.get('step', len(steps))); rets.append(d['eval_return'])
    return steps, rets

def our_curve(log_path):
    steps, rets = [], []
    for line in pathlib.Path(log_path).read_text().splitlines():
        m = re.search(r'\[step (\d+)\] eval return = (-?[\d.]+)', line)
        if m: steps.append(int(m.group(1))); rets.append(float(m.group(2)))
    return steps, rets

rs, rr = ref_curve(f'{BASE}/logdir/pendulum_ref/metrics.jsonl')
plt.figure(figsize=(9,5))
plt.plot(rs, rr, '-o', ms=3, label='reference dreamerv3-torch')

OUR_LOG = ''  # e.g. '/content/Pendulum-v1_20260901-205555(2).log' if you upload ours
if OUR_LOG and pathlib.Path(OUR_LOG).exists():
    os_, or_ = our_curve(OUR_LOG)
    plt.plot(os_, or_, '-s', ms=3, label='ours')

plt.axhline(-200, ls=':', c='gray', label='~solved band (-150..-250)')
plt.xlabel('env step'); plt.ylabel('eval return'); plt.legend(); plt.grid(alpha=.3)
plt.title('Pendulum-v1: reference vs ours (matched hyperparameters)')
plt.show()
print('reference eval_returns:', [round(x,1) for x in rr])

## 7. Watch a video (real env + imagination)

In [ ]:
import glob
from IPython.display import Video, display
BASE = '/content/Dreamerv3/benchmarks/dreamerv3-torch'
envs = sorted(glob.glob(f'{BASE}/videos/pendulum/env_step_*.mp4'))
imgs = sorted(glob.glob(f'{BASE}/videos/pendulum/imag_step_*.mp4'))
if envs: print('real env:', envs[-1]); display(Video(envs[-1], embed=True, width=320))
if imgs: print('imagination:', imgs[-1]); display(Video(imgs[-1], embed=True, width=320))